# Lanzador de la Interfaz Streamlit
PSIB2024Q2-TPI-Grupo7-Ballester Villafañe - Rodriguez Torcelli - Rosado

## 1 · Montar Google Drive

In [5]:
from google.colab import drive
drive.mount('/content/drive')

APP_FOLDER = '/content/drive/MyDrive/AneRBC_App'

import os
os.makedirs(APP_FOLDER, exist_ok=True)
print(f'Carpeta de la app: {APP_FOLDER}')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Carpeta de la app: /content/drive/MyDrive/AneRBC_App


## 2 · Instalar dependencias

In [6]:
!pip install streamlit pyngrok kagglehub openpyxl scikit-image opencv-python-headless -q
print('Dependencias instaladas ✓')

Dependencias instaladas ✓


## 3 · Copiar archivos de la app desde Drive

In [7]:
import shutil, os

# Copia app.py y pipeline.py desde Drive al directorio de trabajo de Colab
for fname in ['app.py', 'pipeline.py']:
    src = os.path.join(APP_FOLDER, fname)
    dst = os.path.join('/content', fname)
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'Copiado: {fname} ✓')
    else:
        print(f'⚠️  No encontrado en Drive: {src}')
        print('   Asegurate de subir app.py y pipeline.py a la carpeta de Drive antes de continuar.')

Copiado: app.py ✓
Copiado: pipeline.py ✓


## 4 · Generar y guardar el modelo de referencia
Esta celda ejecuta el pipeline completo sobre el dataset AneRBC-II para construir
el modelo de referencia y lo guarda como `reference_model.pkl` en Drive.

> **Solo necesitás ejecutar esta celda la primera vez** (o si querés regenerar el modelo).

In [10]:
import sys
sys.path.insert(0, '/content')

import numpy as np
import pandas as pd
import pickle
from pathlib import Path
from PIL import Image
import kagglehub

from pipeline import (
    segment_erythrocytes,
    preprocess_for_analysis,
    extract_cell_features,
    aggregate_image_features,
)

# ── Descarga del dataset ──────────────────────────────────────────────
path = kagglehub.dataset_download(
    'jocelyndumlao/anerbc-anemia-diagnosis-using-rbc-images'
)
print('Dataset path:', path)

DATA_ROOT  = Path(path)
VALID_EXT  = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
matches    = list(DATA_ROOT.rglob('AneRBC_dataset'))
ANERBC_ROOT = matches[0]

image_paths = {'Anemic': [], 'Healthy': []}
for group, label in [('Anemic_individuals', 'Anemic'), ('Healthy_individuals', 'Healthy')]:
    folder = ANERBC_ROOT / 'AneRBC-II' / group / 'Original_images'
    if folder.exists():
        imgs = sorted([p for p in folder.rglob('*') if p.suffix.lower() in VALID_EXT])
        image_paths[label].extend(imgs)
        print(f'AneRBC-II — {label}: {len(imgs)} imágenes')

# ── Procesar imágenes de referencia (250 por clase) ───────────────────
N_REF = 250
records = []

for label, paths in image_paths.items():
    selected = paths[:N_REF]
    print(f'\nProcesando {len(selected)} imágenes de clase {label}…')
    for i, p in enumerate(selected):
        try:
            img_rgb      = np.array(Image.open(p).convert('RGB'))
            result       = segment_erythrocytes(img_rgb)
            img_analysis = preprocess_for_analysis(img_rgb)
            df_cells     = extract_cell_features(
                img_analysis,
                result['labels_features'],
                result['regions_features']
            )
            summary = aggregate_image_features(
                df_cells,
                rbc_count_candidates=len(result['regions_count'])
            )
            summary['label'] = label
            records.append(summary)
        except Exception as e:
            print(f'  Error en {p.name}: {e}')
        if (i + 1) % 10 == 0:
            print(f'  {i+1}/{len(selected)} procesadas')

df_reference = pd.DataFrame(records)
print(f'\nDataset de referencia: {df_reference.shape}')

# ── Construir modelo de referencia ────────────────────────────────────
EXCLUDE_COLS = {'label', 'image_path', 'n_regions_count',
                'n_regions_features', 'image_height', 'image_width'}

feature_cols = [
    c for c in df_reference.columns
    if c not in EXCLUDE_COLS
    and df_reference[c].dtype in [np.float64, np.int64, float, int]
    and df_reference[c].notna().sum() > 10
]

df_ref_clean  = df_reference[feature_cols + ['label']].dropna(subset=feature_cols)
global_mean   = df_ref_clean[feature_cols].mean()
global_std    = df_ref_clean[feature_cols].std().replace(0, 1e-9)
raw_means     = df_ref_clean.groupby('label')[feature_cols].mean()

df_z          = (df_ref_clean[feature_cols] - global_mean) / global_std
df_z['label'] = df_ref_clean['label'].values
centroids     = df_z.groupby('label')[feature_cols].mean()

reference_model = {
    'features'    : feature_cols,
    'global_mean' : global_mean,
    'global_std'  : global_std,
    'raw_means'   : raw_means,
    'centroids'   : centroids,
}

# ── Guardar modelo ────────────────────────────────────────────────────
model_path_drive  = os.path.join(APP_FOLDER, 'reference_model.pkl')
model_path_local  = '/content/reference_model.pkl'

with open(model_path_local, 'wb') as f:
    pickle.dump(reference_model, f)

import shutil
shutil.copy(model_path_local, model_path_drive)
print(f'\nModelo guardado en:')
print(f'  Local  : {model_path_local}')
print(f'  Drive  : {model_path_drive}')
print(f'Features : {len(feature_cols)}')

Using Colab cache for faster access to the 'anerbc-anemia-diagnosis-using-rbc-images' dataset.
Dataset path: /kaggle/input/anerbc-anemia-diagnosis-using-rbc-images
AneRBC-II — Anemic: 6000 imágenes
AneRBC-II — Healthy: 6000 imágenes

Procesando 250 imágenes de clase Anemic…
  10/250 procesadas
  20/250 procesadas
  30/250 procesadas
  40/250 procesadas
  50/250 procesadas
  60/250 procesadas
  70/250 procesadas
  80/250 procesadas
  90/250 procesadas
  100/250 procesadas
  110/250 procesadas
  120/250 procesadas
  130/250 procesadas
  140/250 procesadas
  150/250 procesadas
  160/250 procesadas
  170/250 procesadas
  180/250 procesadas
  190/250 procesadas
  200/250 procesadas
  210/250 procesadas
  220/250 procesadas
  230/250 procesadas
  240/250 procesadas
  250/250 procesadas

Procesando 250 imágenes de clase Healthy…
  10/250 procesadas
  20/250 procesadas
  30/250 procesadas
  40/250 procesadas
  50/250 procesadas
  60/250 procesadas
  70/250 procesadas
  80/250 procesadas
  90/2

## 4b · (Alternativa) Cargar modelo ya guardado desde Drive
Si ya construiste el modelo antes, usá esta celda en lugar de la anterior.

In [11]:
import shutil, pickle

model_drive = os.path.join(APP_FOLDER, 'reference_model.pkl')
model_local = '/content/reference_model.pkl'
shutil.copy(model_drive, model_local)

with open(model_local, 'rb') as f:
    reference_model = pickle.load(f)

print('Modelo cargado desde Drive ✓')
print(f'Features: {len(reference_model["features"])}')

Modelo cargado desde Drive ✓
Features: 94


## 5 · Lanzar la interfaz Streamlit
Necesitás un token de ngrok. Obtené uno gratis en https://ngrok.com (es gratuito).
Pegalo en la variable `NGROK_TOKEN` abajo.

In [12]:
import subprocess, threading, time
from pyngrok import ngrok, conf

# ─── Pegá tu token de ngrok aquí ──────────────────────────────────────
NGROK_TOKEN = '3EpUITY27BNFjenSwss8Uz910zx_5NEvjZ9TPRQ5GbMZLCyJi'
# ──────────────────────────────────────────────────────────────────────

conf.get_default().auth_token = NGROK_TOKEN

# Lanzar Streamlit en background
def run_streamlit():
    subprocess.run(
        ['streamlit', 'run', '/content/app.py',
         '--server.port', '8501',
         '--server.headless', 'true',
         '--server.enableCORS', 'false'],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

thread = threading.Thread(target=run_streamlit, daemon=True)
thread.start()
time.sleep(5)  # Esperar que Streamlit inicie

# Túnel ngrok
public_url = ngrok.connect(8501)
print('─' * 60)
print('🚀 Interfaz disponible en:')
print(f'   {public_url}')
print('─' * 60)
print('Abrí el link de arriba en tu navegador.')
print('El túnel se cierra cuando interrumpís o cerrás Colab.')

────────────────────────────────────────────────────────────
🚀 Interfaz disponible en:
   NgrokTunnel: "https://oval-numbing-unfrozen.ngrok-free.dev" -> "http://localhost:8501"
────────────────────────────────────────────────────────────
Abrí el link de arriba en tu navegador.
El túnel se cierra cuando interrumpís o cerrás Colab.


## 6 · (Opcional) Actualizar archivos en Drive
Si modificás `app.py` o `pipeline.py` localmente, usá esta celda para sincronizarlos con Drive.

In [ ]:
import shutil
for fname in ['app.py', 'pipeline.py']:
    shutil.copy(f'/content/{fname}', os.path.join(APP_FOLDER, fname))
    print(f'Sincronizado: {fname} → Drive ✓')